# Create Final Baseline Report (Notebook)

Notebook version of the baseline report pipeline, organized into executable cells with comments.

This notebook mirrors `create_final_process_baseline_report.py` and is intended for transparent, step-by-step execution in VS Code.

## 1) Set Up Notebook Environment and Paths

Define project paths, output locations, and runtime toggles.

> To switch environments (local/dev/prod), change `ROOT` and/or the file names below.

In [ ]:
# --- Path configuration for notebook execution ---
# ROOT points to repository root. Update this if your local checkout path differs.
import os

ROOT = "C:/Users/takbh/auracheck"
OUT_DIR = os.path.join(ROOT, "baseline", "outputs", "final_baseline_model")
DATA_PATH = "C:/Users/takbh/auracheck/Dataset/students_mental_health_survey_with_burnout_final.csv"

# --- Supabase table link + data fetch (commented by default) ---
# Matches your .env keys: SUPABASE_URL, SUPABASE_ANON_KEY, SUPABASE_SERVICE_ROLE_KEY
# SUPABASE_URL = os.getenv("SUPABASE_URL")
# SUPABASE_TABLE_NAME = "students_mental_health_survey_with_burnout_final"
# SUPABASE_TABLE_URL = f"{SUPABASE_URL}/rest/v1/{SUPABASE_TABLE_NAME}"
# SUPABASE_KEY = (os.getenv("SUPABASE_ANON_KEY") or os.getenv("SUPABASE_SERVICE_ROLE_KEY") or "").strip()
#
# Uncomment the following lines to fetch data from Supabase instead of local CSV.
# from supabase import create_client
# supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
# df_raw = pd.DataFrame(
#     supabase.table(SUPABASE_TABLE_NAME)
#     .select("*")
#     .limit(10000)
#     .execute()
#     .data
# )


# Output artifacts generated by the report pipeline.
OUT_PDF = os.path.join(OUT_DIR, "final_process_baseline_comparison_report.pdf")
OUT_TXT = os.path.join(OUT_DIR, "final_process_baseline_comparison_report_summary.txt")
OUT_CM = os.path.join(OUT_DIR, "final_selected_baseline_confusion_matrix.csv")
OUT_SENS_SPEC = os.path.join(OUT_DIR, "final_selected_baseline_sensitivity_specificity.csv")

# Runtime flags for notebook behavior.
OVERWRITE_OUTPUTS = True
SHOW_HEAD = 5

os.makedirs(OUT_DIR, exist_ok=True)
print("ROOT:", ROOT)
print("DATA_PATH exists:", os.path.exists(DATA_PATH))
print("OUT_DIR:", OUT_DIR)

## 2) Import Libraries and Configure Display/Logging

Import the same dependencies used by the script and configure notebook-friendly display/logging.

In [ ]:
# Core scientific/data stack used in the report pipeline.
import json
import logging
import textwrap
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Display/log settings help with debugging and readable notebook outputs.
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
warnings.filterwarnings("ignore", category=UserWarning)

logging.basicConfig(level=logging.INFO, format="[%(levelname)s] %(message)s")
logger = logging.getLogger("baseline-report")
logger.info("Libraries imported and display/logging configured.")

## 3) Load Inputs and Baseline Data Sources

Load baseline dataset and quick schema checks to confirm expected structure.

In [ ]:
# Load dataset used by the baseline model (default: local CSV).
df_raw = pd.read_csv(DATA_PATH)

# If you enabled Supabase fetch in Cell 3, comment the line above and use this instead:
# df_raw = df_raw  # already loaded from Supabase in Cell 3

# Expected key target-related columns.
expected_cols = {"burnout_raw_score"}
missing_required = sorted(list(expected_cols - set(df_raw.columns)))
if missing_required:
    raise ValueError(f"Dataset is missing required columns: {missing_required}")

print("Shape:", df_raw.shape)
print("First columns:", list(df_raw.columns[:12]))
print("Null counts (top 10):")
print(df_raw.isna().sum().sort_values(ascending=False).head(10))

df_raw.head(SHOW_HEAD)

## 4) Define Reusable Helper Functions (Commented)

These helpers mirror script behavior for preprocessing, diagnostics, and report support.

In [ ]:
# Class names used across confusion matrix, metrics, and reporting.
CLASS_NAMES = ["Very Low (Q1)", "Low (Q2)", "Moderate (Q3)", "High (Q4)"]

# Final baseline predictor set after multicollinearity pruning.
FEATURES_PRUNED = [
    "Course", "Gender", "Sleep_Quality", "Physical_Activity", "Diet_Quality",
    "Social_Support", "Relationship_Status", "Substance_Use", "Counseling_Service_Use",
    "Family_History", "Chronic_Illness", "Financial_Stress",
    "Extracurricular_Involvement", "Residence_Type"
]

# Fixed mappings ensure deterministic categorical encoding at train + inference time.
ENCODING_MAP = {
    "Gender": {"Female": 0, "Male": 1},
    "Sleep_Quality": {"Poor": 0, "Average": 1, "Good": 2},
    "Physical_Activity": {"Low": 0, "Moderate": 1, "High": 2},
    "Diet_Quality": {"Good": 0, "Average": 1, "Poor": 2},
    "Social_Support": {"High": 0, "Moderate": 1, "Low": 2},
    "Substance_Use": {"Never": 0, "Unknown": 1, "Occasionally": 2, "Frequently": 3},
    "Counseling_Service_Use": {"Never": 0, "Occasionally": 1, "Frequently": 2},
    "Family_History": {"No": 0, "Yes": 1},
    "Chronic_Illness": {"No": 0, "Yes": 1},
    "Extracurricular_Involvement": {"High": 0, "Moderate": 1, "Low": 2},
    "Course": {"Business": 0, "Computer Science": 1, "Engineering": 2, "Law": 3, "Medical": 4, "Others": 5},
    "Relationship_Status": {"In a Relationship": 0, "Married": 1, "Single": 2},
    "Residence_Type": {"Off-Campus": 0, "On-Campus": 1, "With Family": 2},
}


def preprocess(df: pd.DataFrame, features: list) -> pd.DataFrame:
    """Preprocess selected features.

    Steps:
    1) Select model features
    2) Impute missing values
       - categorical -> 'Unknown'
       - numeric -> median
    3) Apply deterministic categorical encodings
    4) Return numeric DataFrame ready for scaling
    """
    X = df[features].copy()
    for c in X.columns:
        if X[c].isnull().any():
            if X[c].dtype == object:
                X[c] = X[c].fillna("Unknown")
            else:
                X[c] = X[c].fillna(X[c].median())
    for c, mapping in ENCODING_MAP.items():
        if c in X.columns:
            X[c] = X[c].astype(str).map(mapping).fillna(1)
    return X.astype(float)


def make_quartile_target(df: pd.DataFrame) -> pd.Series:
    """Create 4-class target from burnout_raw_score quartiles."""
    return pd.qcut(df["burnout_raw_score"].astype(float), q=4, labels=[0, 1, 2, 3], duplicates="drop").astype(int)


def safe_read_csv(path: str):
    """Read CSV if it exists, otherwise return None."""
    return pd.read_csv(path) if os.path.exists(path) else None


logger.info("Helper functions initialized.")

## 5) Build Core Baseline Metrics Pipeline

Train/evaluate the balanced multinomial baseline on a stratified holdout split.

In [ ]:
# Build model-ready X/y.
X = preprocess(df_raw, FEATURES_PRUNED)
y = make_quartile_target(df_raw)

# Stratified split preserves class proportions in train/test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize for stable optimization and comparable coefficient scale.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Balanced multinomial logistic regression baseline.
clf = LogisticRegression(
    solver="lbfgs",
    max_iter=5000,
    random_state=42,
    class_weight="balanced",
)
clf.fit(X_train_s, y_train)

# Predictions and probabilities for metrics and report visuals.
y_pred = clf.predict(X_test_s)
y_proba = clf.predict_proba(X_test_s)

# Core metrics used in the report.
acc = (y_pred == y_test).mean()
macro_f1 = f1_score(y_test, y_pred, average="macro", labels=[0, 1, 2, 3], zero_division=0)
ll = log_loss(y_test, y_proba, labels=[0, 1, 2, 3])

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Log Loss: {ll:.4f}")

## 6) Generate Final Baseline Report Tables

Create report-ready tables for performance and confusion diagnostics.

In [ ]:
# Confusion matrix table.
cm = confusion_matrix(y_test, y_pred, labels=[0, 1, 2, 3])
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)

# Per-class sensitivity/specificity table.
rows = []
total = cm.sum()
for i, cname in enumerate(CLASS_NAMES):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = total - tp - fn - fp
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    rows.append(
        {
            "Class": cname,
            "Sensitivity_Recall": float(sensitivity),
            "Specificity": float(specificity),
            "TP": int(tp),
            "FN": int(fn),
            "FP": int(fp),
            "TN": int(tn),
        }
    )
rates_df = pd.DataFrame(rows)

# Performance summary table used by the report page.
perf_df = pd.DataFrame([
    {
        "Accuracy": float(acc),
        "Macro_F1": float(macro_f1),
        "Log_Loss": float(ll),
    }
])

cm_df, rates_df.head(), perf_df

## 7) Create Visual Summaries for the Report

Generate key chart components used in the PDF (coefficient heatmap and confusion matrix).

In [ ]:
# Coefficient heatmap visualizes directional influence per class-feature pair.
coef_df = pd.DataFrame(clf.coef_, index=CLASS_NAMES, columns=FEATURES_PRUNED)
plt.figure(figsize=(10, 4))
sns.heatmap(coef_df, cmap="coolwarm", center=0, cbar_kws={"label": "Coefficient"})
plt.title("Multinomial Coefficient Heatmap")
plt.xlabel("Features")
plt.ylabel("Classes")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Confusion matrix visual for classification error structure by class.
plt.figure(figsize=(6.5, 5.2))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Baseline Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

## 8) Export Report Artifacts and Save Outputs

Use the maintained script entrypoint for deterministic export naming and full PDF generation.

In [ ]:
# Import and run the authoritative script implementation.
# This avoids drift between notebook and production report logic.
import create_final_process_baseline_report as report_builder

# Run full report generation pipeline.
report_builder.main()

# Confirm key artifacts exist.
for f in [
    "final_process_baseline_comparison_report.pdf",
    "final_process_baseline_comparison_report_summary.txt",
    "final_selected_baseline_confusion_matrix.csv",
    "final_selected_baseline_sensitivity_specificity.csv",
]:
    fp = os.path.join(OUT_DIR, f)
    print(f"{f}:", os.path.exists(fp))

## 9) Add Validation Checks and Reproducibility Controls

Assertions and quick checks to verify output completeness and stable reruns.

In [ ]:
# Validate that required outputs exist and are non-empty.
required_outputs = [
    OUT_PDF,
    OUT_TXT,
    OUT_CM,
    OUT_SENS_SPEC,
]

for fp in required_outputs:
    assert os.path.exists(fp), f"Missing output: {fp}"
    assert os.path.getsize(fp) > 0, f"Empty output file: {fp}"

# Additional integrity checks for tabular outputs.
cm_check = pd.read_csv(OUT_CM, index_col=0)
rates_check = pd.read_csv(OUT_SENS_SPEC)

assert cm_check.shape == (4, 4), "Confusion matrix must be 4x4."
assert {"Class", "Sensitivity_Recall", "Specificity"}.issubset(rates_check.columns), "Rates CSV schema mismatch."

print("Validation passed. Outputs are present and structurally valid.")
print("Tip: For reproducibility, keep random_state=42 and the same dataset version.")